In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [6]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [8]:
brochure_system_prompt = """
You are an expert corporate copywriter. 
Generate a comprehensive, professional business brochure based on the provided website data. 

CRITICAL STRUCTURE RULE:
1. Generate the complete brochure in English first.
2. At the very end of the English brochure, add a horizontal line separator (---).
3. Right below the separator, create a header "## النسخة العربية (Arabic Version)".
4. Provide the exact same brochure completely translated into high-quality, professional Arabic.
Maintain the same markdown formatting (bullet points, bold text) in both sections.
"""

In [9]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.
Provide the full brochure in English, then add '---', and write the exact same brochure in Arabic.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [10]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [11]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face  
**The AI Community Building the Future**

---

## About Us  
Hugging Face is the premier platform where the global machine learning community converges to create, discover, and collaborate on artificial intelligence models, datasets, and applications. As an open-source hub, Hugging Face accelerates innovation by providing an ecosystem for both individual developers and enterprises to share resources and expertise.

---

## Our Platform Highlights

- **Models:** Access and browse over 2 million cutting-edge AI models across multiple domains and tasks. Stay updated with the latest trending models regularly refreshed by the community.
  
- **Datasets:** Explore more than 500,000 datasets supporting a wide array of machine learning projects, from natural language processing to computer vision.
  
- **Spaces:** Run and share AI-powered applications and demos effortlessly with Spaces—an environment to showcase and test models in live settings.
  
- **Buckets:** Secure and scalable storage solutions designed to support machine learning workflows by hosting model data and artifacts efficiently.

---

## Community & Collaboration

- Hugging Face fosters a vibrant community where researchers, engineers, and AI enthusiasts engage through forums, Discord, and GitHub.
- Participate in collaborative open-source projects and gain insights from daily posts, blogs, and research papers.
- Access comprehensive documentation and learn resources tailored for both beginners and advanced users.
- Benefit from enterprise-grade solutions like Hugging Face PRO, tailored support, and inference endpoints for seamless deployment.

---

## Why Choose Hugging Face?

- **Innovation at Scale:** Stay ahead with an ever-evolving repository of models and datasets that push the AI frontier.
- **Open Source Excellence:** Leverage a robust open-source stack that empowers developers to build and deploy AI faster.
- **Enterprise Ready:** Customized solutions and support to fit the needs of organizations scaling AI initiatives.
- **Community Driven:** Engage with a global network of passionate AI professionals and contributors driving ethical and impactful AI development.

---

## Join Us  
Discover AI applications, explore extensive models and datasets, and become part of a pioneering community shaping the future of machine learning.

- Explore AI Apps  
- Browse 2M+ Models  
- Access 500K+ Datasets  
- Connect with Experts Worldwide

**Hugging Face** — The Home of Machine Learning Collaboration

---

## Contact & Access  
- Website: [huggingface.co](https://huggingface.co)  
- Community Forum, Discord, GitHub for real-time support and collaboration  
- Enterprise solutions and support offerings available  

---

# النسخة العربية (Arabic Version)  

# Hugging Face  
**مجتمع الذكاء الاصطناعي لبناء المستقبل**

---

## من نحن  
هغينغ فيس هو المنصة الرائدة التي يجتمع فيها مجتمع التعلم الآلي العالمي لإنشاء واكتشاف والتعاون في نماذج الذكاء الاصطناعي، مجموعات البيانات، والتطبيقات. بفضل كونه مركزًا مفتوح المصدر، يسرع هغينغ فيس الابتكار من خلال توفير نظام بيئي للأفراد والشركات لمشاركة الموارد والخبرات.

---

## أبرز ميزات منصتنا

- **النماذج:** الولوج وتصفح أكثر من 2 مليون نموذج ذكاء اصطناعي متطور عبر مجالات ومهام متعددة. تحديثات مستمرة للنماذج الرائجة بفضل المجتمع.
  
- **مجموعات البيانات:** استكشاف أكثر من 500,000 مجموعة بيانات تدعم مشاريع التعلم الآلي المتنوعة، من معالجة اللغة الطبيعية إلى رؤية الكمبيوتر.
  
- **المساحات:** تشغيل ومشاركة التطبيقات والعروض التوضيحية المدعومة بالذكاء الاصطناعي بسهولة من خلال المساحات، بيئة لعرض واختبار النماذج بشكل مباشر.
  
- **الحاويات:** حلول تخزين آمنة وقابلة للتوسع مصممة لدعم سير عمل التعلم الآلي عبر استضافة بيانات النماذج والمنتجات بكفاءة.

---

## المجتمع والتعاون

- يدعم هغينغ فيس مجتمعًا نابضًا حيث يتفاعل الباحثون والمهندسون وعشاق الذكاء الاصطناعي عبر المنتديات، ديسكورد، وجيتهاب.
- شارك في مشاريع مفتوحة المصدر تعاونية واحصل على رؤى من المشاركات اليومية، المدونات، والأوراق البحثية.
- وصول إلى توثيقات شاملة وموارد تعليمية مخصصة للمبتدئين والمحترفين.
- استفد من حلول الأعمال المؤسسية مثل Hugging Face PRO، والدعم المخصص، ونقاط نهاية الاستدلال لنشر متكامل.

---

## لماذا تختار Hugging Face؟

- **الابتكار على نطاق واسع:** كن السبّاق مع مجموعة متطورة ومتجددة من النماذج وبيانات التدريب التي تدفع حدود الذكاء الاصطناعي.
- **تميّز مفتوح المصدر:** استغل بنية مفتوحة قوية تمكّن المطورين من البناء والنشر السريع لتطبيقات الذكاء الاصطناعي.
- **جاهزية المؤسسات:** حلول ودعم مخصص يلبّي احتياجات المؤسسات في تعظيم مبادرات الذكاء الاصطناعي.
- **القيادة المجتمعية:** تواصل مع شبكة عالمية من المهنيين والمساهمين الشغوفين الذين يدفعون إلى تطوير ذكاء اصطناعي أخلاقي وفعّال.

---

## انضم إلينا  
اكتشف تطبيقات الذكاء الاصطناعي، تصفح مجموعات هائلة من النماذج والبيانات، وكن جزءًا من مجتمع رائد يشكل مستقبل التعلم الآلي.

- استكشاف تطبيقات الذكاء الاصطناعي  
- تصفح أكثر من 2 مليون نموذج  
- الوصول إلى أكثر من 500 ألف مجموعة بيانات  
- التواصل مع خبراء عالميين  

**Hugging Face** — موطن التعاون في مجال التعلم الآلي

---

## الاتصال والوصول  
- الموقع الإلكتروني: [huggingface.co](https://huggingface.co)  
- منتدى المجتمع، ديسكورد، وجيتهاب للدعم والتعاون الفوري  
- حلول الشركات وعروض الدعم متاحة  

